In [1]:
# Load libraries
import pandas as pd
import os
import numpy as np
from pathlib import Path

import geopandas as gpd
data_dir = Path(".") 

In [2]:
########################################
# Raw crime data preparation
########################################

# Crime data
crime = pd.read_csv(data_dir / "MPS Ward Level Crime (Historical).csv")

# Make column names strings
crime.columns = crime.columns.astype(str)

# Select 2022/23 financial year: Apr 2022 to Mar 2023
crime_2223_cols = [
    "202204", "202205", "202206", "202207", "202208", "202209",
    "202210", "202211", "202212", "202301", "202302", "202303"
]

# Check missing columns
missing_cols = [col for col in crime_2223_cols if col not in crime.columns]
print("Missing columns:", missing_cols)

# Sum monthly crime counts
crime["crime_2022_23"] = crime[crime_2223_cols].sum(axis=1)

# Aggregate to ward level
crime_ward_2223 = (
    crime.groupby(["WardCode", "WardName"], as_index=False)["crime_2022_23"]
    .sum()
)

# Preview result
crime_ward_2223.head()

Missing columns: []


,WardCode,WardName,crime_2022_23
0,E05009317,Bethnal Green,2480
1,E05009318,Blackwall & Cubitt Town,1568
2,E05009319,Bow East,1984
3,E05009320,Bow West,1433
4,E05009321,Bromley North,1131


In [3]:
########################################
# Socio-economic and demographic variables
########################################

########################################
# 1. Population density
########################################

# Read 2021 population data
pop_2021 = pd.read_excel(data_dir / "Usual Residents.xlsx", sheet_name="2021")

# Keep ward code, ward name and population
pop = pop_2021[[
    "ward code",
    "All usual residents"
]].copy()

# Rename columns
pop = pop.rename(columns={
    "ward code": "WardCode",
    "All usual residents": "population_2021"
})

# Keep ward-level rows only
pop = pop[pop["WardCode"].astype(str).str.startswith("E050")].copy()

# Read ward boundary data
wards = gpd.read_file(
    data_dir / "Wards_December_2022_Boundaries_UK_BGC_5935341910977814913.geojson"
)

# Keep only London wards that match the population data
wards_london = wards[wards["WD22CD"].isin(pop["WardCode"])].copy()

# Check matched wards
print("Matched London wards:", len(wards_london))

# Convert to British National Grid for area calculation
wards_london = wards_london.to_crs(epsg=27700)

# Calculate ward area in square kilometres
wards_london["area_km2"] = wards_london.geometry.area / 1_000_000

# Keep ward code and area
ward_area = wards_london[[
    "WD22CD",
    "area_km2"
]].copy()

# Rename ward code
ward_area = ward_area.rename(columns={
    "WD22CD": "WardCode"
})

# Merge population with ward area
pop_density = pop.merge(
    ward_area,
    on="WardCode",
    how="left"
)

# Calculate population density: persons per km2
pop_density["pop_density_2021"] = (
    pop_density["population_2021"] / pop_density["area_km2"]
).round(2)

# Preview result
pop_density.head()

Matched London wards: 679


,WardCode,population_2021,area_km2,pop_density_2021
0,E05014053,3965,0.462382,8575.16
1,E05014054,9890,1.443242,6852.63
2,E05014055,10060,4.428038,2271.89
3,E05014056,8355,2.757906,3029.47
4,E05014057,9790,0.916854,10677.82


In [4]:
########################################
# 2. Young adult population share
########################################

# Calculate the proportion of young adults aged 20–39

# Read 2021 age data
age_2021 = pd.read_excel(data_dir / "Five year age bands.xlsx", sheet_name="2021")

# Keep ward code, total population, and 20-39 age groups
age = age_2021[[
    "ward code",
    "All usual residents",
    "Aged 20 to 24 years",
    "Aged 25 to 29 years",
    "Aged 30 to 34 years",
    "Aged 35 to 39 years"
]].copy()

# Rename columns
age = age.rename(columns={
    "ward code": "WardCode",
    "All usual residents": "population_2021"
})

# Keep ward-level rows only
age = age[age["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate number of young adults aged 20-39
age["young_adult_20_39"] = (
    age["Aged 20 to 24 years"] +
    age["Aged 25 to 29 years"] +
    age["Aged 30 to 34 years"] +
    age["Aged 35 to 39 years"]
)

# Calculate young adult share
age["young_adult_share_2021"] = (
    age["young_adult_20_39"] / age["population_2021"] * 100).round(2)


# Keep final variable
young_adult = age[[
    "WardCode",
    "young_adult_share_2021"
]].copy()

# Preview result
young_adult.head()

,WardCode,young_adult_share_2021
1,E05014053,42.24
2,E05014054,29.23
3,E05014055,33.07
4,E05014056,26.98
5,E05014057,27.85


In [5]:
########################################
# 3. Unemployment
########################################

# Calculate the proportion of young adults aged 20–39
# Read 2021 economic activity data
econ_2021 = pd.read_excel(data_dir / "Economic Activity.xlsx", sheet_name="2021")

# Keep useful columns
econ = econ_2021[[
    "ward code",
    "Economically active Total",
    "Unemployed"
]].copy()

# Rename ward code
econ = econ.rename(columns={
    "ward code": "WardCode"
})

# Keep ward-level rows only
econ = econ[econ["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate unemployment rate
econ["unemployment_rate_2021"] = (
    econ["Unemployed"] / econ["Economically active Total"] * 100
).round(2)

# Keep final variable
unemployment = econ[[
    "WardCode",
    "unemployment_rate_2021"
]].copy()

# Preview result
unemployment.head()

,WardCode,unemployment_rate_2021
1,E05014053,6.99
2,E05014054,6.97
3,E05014055,7.26
4,E05014056,7.59
5,E05014057,8.20


In [6]:
########################################
# 4. Higher education qualification share
########################################

# Calculate the proportion of residents aged 16 and over with Level 4 qualifications or above.
# Level 4 or above is used as an indicator of higher educational attainment.

# Read 2021 qualifications data
qual_2021 = pd.read_excel(data_dir / "qualifications.xlsx", sheet_name="2021")

# Keep useful columns
qual = qual_2021[[
    "ward code",
    "Usual residents aged 16+",
    "Level 4+"
]].copy()

# Rename ward code
qual = qual.rename(columns={
    "ward code": "WardCode"
})

# Keep ward-level rows only
qual = qual[qual["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate higher education share
qual["higher_education_share_2021"] = (
    qual["Level 4+"] / qual["Usual residents aged 16+"] * 100
).round(2)

# Keep final variable only
higher_education = qual[[
    "WardCode",
    "higher_education_share_2021"
]].copy()

# Preview result
higher_education.head()

,WardCode,higher_education_share_2021
1,E05014053,48.80
2,E05014054,31.18
3,E05014055,43.73
4,E05014056,31.32
5,E05014057,30.39


In [7]:
########################################
# 5. Migration background
########################################

# Calculate the proportion of residents who were born outside the UK.

# Read 2021 country of birth data
birth_2021 = pd.read_excel(data_dir / "Country of birth.xlsx", sheet_name="2021")

# Keep useful columns
birth = birth_2021[[
    "ward code",
    "All Usual residents",
    "United Kingdom"
]].copy()

# Rename ward code
birth = birth.rename(columns={
    "ward code": "WardCode"
})

# Keep ward-level rows only
birth = birth[birth["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate non-UK-born share
birth["non_uk_born_share_2021"] = (
    (birth["All Usual residents"] - birth["United Kingdom"]) 
    / birth["All Usual residents"] * 100
).round(2)

# Keep final variable only
non_uk_born = birth[[
    "WardCode",
    "non_uk_born_share_2021"
]].copy()

# Preview result
non_uk_born.head()


,WardCode,non_uk_born_share_2021
1,E05014053,65.72
2,E05014054,39.82
3,E05014055,43.87
4,E05014056,43.58
5,E05014057,40.53


In [8]:
########################################
# 6. Lone-parent household share
########################################

# Calculate the proportion of households that are lone-parent households.
# This variable is used as an indicator of household structure at ward level.

# Read 2021 household composition data
hh_2021 = pd.read_excel(data_dir / "Household composition.xlsx", sheet_name="2021")

# Rename ward code
hh_2021 = hh_2021.rename(columns={
    "ward code": "WardCode"
})

# Keep useful columns
hh = hh_2021[[
    "WardCode",
    "All households",
    "Lone parent: dependent children",
    "Lone parent: non-dependent children"
]].copy()

# Keep ward-level rows only
hh = hh[hh["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate lone-parent households
hh["lone_parent_households"] = (
    hh["Lone parent: dependent children"] +
    hh["Lone parent: non-dependent children"]
)

# Calculate lone-parent household share
hh["lone_parent_household_share_2021"] = (
    hh["lone_parent_households"] / hh["All households"] * 100
).round(2)

# Keep final variable only
lone_parent = hh[[
    "WardCode",
    "lone_parent_household_share_2021"
]].copy()

# Preview result
lone_parent.head()

,WardCode,lone_parent_household_share_2021
1,E05014053,11.03
2,E05014054,20.60
3,E05014055,21.23
4,E05014056,21.61
5,E05014057,18.30


In [9]:
########################################
# 7. Household deprivation share
########################################

# Calculate the proportion of households deprived in two or more dimensions.
# This variable is used as an indicator of household-level deprivation at ward level.

# Read 2021 household deprivation data
dep_2021 = pd.read_excel(data_dir / "Household deprivation.xlsx", sheet_name="2021")

# Keep useful columns
dep = dep_2021[[
    "ward code",
    "All Households",
    "2 dimensions",
    "3 dimensions",
    "4 dimensions"
]].copy()

# Rename ward code
dep = dep.rename(columns={
    "ward code": "WardCode"
})

# Keep ward-level rows only
dep = dep[dep["WardCode"].astype(str).str.startswith("E050")].copy()

# Calculate severe household deprivation
dep["severe_household_deprivation_share_2021"] = (
    (dep["2 dimensions"] + dep["3 dimensions"] + dep["4 dimensions"])
    / dep["All Households"] * 100
).round(2)

# Keep final variable only
household_deprivation = dep[[
    "WardCode",
    "severe_household_deprivation_share_2021"
]].copy()

# Preview result
household_deprivation.head()

,WardCode,severe_household_deprivation_share_2021
1,E05014053,19.73
2,E05014054,23.80
3,E05014055,19.84
4,E05014056,22.96
5,E05014057,27.86


In [10]:
# Construct final modelling dataset

# Start with ward-level crime data
model_df = crime_ward_2223.copy()

# Merge population and population density
model_df = model_df.merge(
    pop_density[[
        "WardCode",
        "population_2021",
        "area_km2",
        "pop_density_2021"
    ]],
    on="WardCode",
    how="left"
)

# Calculate crime rate per 1,000 residents
model_df["crime_rate_1000"] = (
    model_df["crime_2022_23"] / model_df["population_2021"] * 1000
).round(2)

# Merge young adult share
model_df = model_df.merge(
    young_adult,
    on="WardCode",
    how="left"
)

# Merge unemployment rate
model_df = model_df.merge(
    unemployment,
    on="WardCode",
    how="left"
)

# Merge higher education share
model_df = model_df.merge(
    higher_education,
    on="WardCode",
    how="left"
)

# Merge non-UK-born share
model_df = model_df.merge(
    non_uk_born,
    on="WardCode",
    how="left"
)

# Merge lone-parent household share
model_df = model_df.merge(
    lone_parent,
    on="WardCode",
    how="left"
)

# Merge severe household deprivation share
model_df = model_df.merge(
    household_deprivation,
    on="WardCode",
    how="left"
)

In [11]:
# Final dataset checks
# Drop columns not used for modelling
model_df = model_df.drop(columns=["area_km2"])

# Check missing values
model_df.isna().sum()

# Preview final dataset
model_df.head()


,WardCode,WardName,crime_2022_23,population_2021,pop_density_2021,crime_rate_1000,young_adult_share_2021,unemployment_rate_2021,higher_education_share_2021,non_uk_born_share_2021,lone_parent_household_share_2021,severe_household_deprivation_share_2021
0,E05009317,Bethnal Green,2480,21229,17558.27,116.82,42.79,7.12,44.70,41.56,13.35,26.50
1,E05009318,Blackwall & Cubitt Town,1568,21834,16143.67,71.81,57.44,6.59,62.72,56.55,7.66,13.71
2,E05009319,Bow East,1984,19530,10453.38,101.59,50.32,6.72,54.30,36.07,11.58,20.99
3,E05009320,Bow West,1433,13708,10091.07,104.54,44.24,5.18,52.26,35.20,10.47,22.46
4,E05009321,Bromley North,1131,11429,19047.46,98.96,43.48,8.95,43.15,43.98,13.64,26.67


In [12]:
# Export final modelling dataset
model_df.to_csv(data_dir / "model_df.csv", index=False)

print("Saved to:", data_dir / "model_df.csv")

Saved to: model_df.csv
